<a href="https://colab.research.google.com/github/samjurassic/datascience-demo/blob/main/coda/HBS_CoDA_Python_Part3_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HBS CoDA Python Workshop - Part 3

Learning objectives include:

* Run OLS models in Python

* Prove the Frisch-Waugh-Lovell theorem

* Learn double/debiased machine learning framework for casual inference

* Estimate average treatment effect of 401k participation on savings using DoubleML

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from graphviz import Digraph
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

## Linear Regression

In [ ]:
# --- 1. Simulate the Data ---
np.random.seed(123)
n = 400

# X: Years of Experience (between 0 and 20 years)
# Y: Salary
# The True Math: Base salary is $50k. Every year of experience adds $4,500.
# We add "noise" because real life is messy (negotiation skills, different companies, etc.)
experience = np.random.uniform(0, 20, size=n)
noise = np.random.normal(0, 15000, size=n)

# Create the DataFrame
df = pd.DataFrame({
    'experience': experience,
    'salary': 50000 + (4500 * experience) + noise,
})

display(df.head())
display(df.corr())
sns.scatterplot(df, x="experience", y="salary", alpha=0.6, color="royalblue")

In [ ]:
print("--- OLS REGRESSION RESULTS ---")
# --- 2. Run the OLS Model ---
# sm.add_constant creates the "Intercept" (the starting salary at 0 years)
X = sm.add_constant(df[["experience"]])
model = sm.OLS(df["salary"], X).fit()

# Print just the coefficients table as requested
print(model.summary().tables[1])

In [ ]:
# --- 3. Visualizing with Manually Fitted Line ---
plt.figure(figsize=(9, 6))

# Step A: Plot the raw data points using a basic scatterplot
sns.scatterplot(
    x="experience",
    y="salary",
    data=df,
    color="royalblue",
    alpha=0.6,
    s=60, # Dot size
    edgecolor="w",
    label="Actual Data (Observed)"
)

# Step B: Generate the predictions from our OLS model
# To draw a clean line, we need to sort our X values from lowest to highest
df_sorted = df.sort_values(by="experience")
X_sorted = sm.add_constant(df_sorted[["experience"]])

# Get the model's predicted Y values for those sorted X values
predicted_salary = model.predict(X_sorted)

# Step C: Overlay the fitted line
plt.plot(
    df_sorted["experience"],
    predicted_salary,
    color="darkred",
    linewidth=3,
    label="OLS Fitted Line (Predicted)"
)

# Formatting the plot
plt.title("Simple OLS: Years of Experience vs. Salary", fontsize=16, pad=15)
plt.xlabel("Years of Experience", fontsize=12)
plt.ylabel("Annual Salary ($)", fontsize=12)
plt.legend(loc="upper left", fontsize=11)
plt.grid(True, linestyle='--', alpha=0.5)

# Extract equation values dynamically for the text box
intercept = model.params['const']
slope = model.params['experience']
plt.text(0.5, 120000, f"Predicted Salary = ${intercept:,.0f} + (${slope:,.0f} * Years)",
         fontsize=11, bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray'))

plt.ylim(20000, 150000)
plt.show()

### Difference-in-Differences

Difference-in-Differences (DiD) estimates causal effects by comparing the change in outcomes over time between a treated group and a control group. The key assumption is **parallel trends**: absent treatment, both groups would have followed the same trajectory.

In [ ]:
np.random.seed(42)

# --- Simulate data ---
# Two groups: treated (got a job training program) and control
# Two periods: before and after the program
n = 200  # per group-period

treated = np.concatenate([np.ones(n), np.ones(n), np.zeros(n), np.zeros(n)])
post = np.concatenate([np.zeros(n), np.ones(n), np.zeros(n), np.ones(n)])

true_did_effect = 5.0

# Outcome: earnings
# Treated group starts higher (baseline diff = 3), both groups trend up by 2
earnings = (
    20
    + 3 * treated                    # treated group has higher baseline
    + 2 * post                       # common time trend
    + true_did_effect * treated * post  # causal effect of the program
    + np.random.normal(0, 2, 4 * n)
)

df = pd.DataFrame({"treated": treated, "post": post, "earnings": earnings})

# --- DiD regression: Y = b0 + b1*Treated + b2*Post + b3*(Treated x Post) + e ---
df["treated_x_post"] = df["treated"] * df["post"]
X = sm.add_constant(df[["treated", "post", "treated_x_post"]])
model = sm.OLS(df["earnings"], X).fit()

print(model.summary().tables[1])
print(f"\nTrue DiD effect:      {true_did_effect:.2f}")
print(f"Estimated DiD effect: {model.params['treated_x_post']:.2f}")

# --- Plot group means ---
means = df.groupby(["treated", "post"])["earnings"].mean().unstack()
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([0, 1], means.loc[0], "o-", label="Control", color="steelblue")
ax.plot([0, 1], means.loc[1], "o-", label="Treated", color="coral")
# Counterfactual
ax.plot([0, 1], [means.loc[1, 0], means.loc[1, 0] + (means.loc[0, 1] - means.loc[0, 0])],
        "o--", color="coral", alpha=0.4, label="Treated counterfactual")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Pre", "Post"])
ax.set_ylabel("Earnings")
ax.set_title("Difference-in-Differences")
ax.legend()
plt.tight_layout()
plt.show()

### Frisch-Waugh-Lovell Theorem

The Frisch-Waugh-Lovell (FWL) theorem states that in a multiple regression of Y on X₁ and X₂, the coefficient on X₁ is identical to the coefficient from a simple regression of the residuals of Y (after removing X₂) on the residuals of X₁ (after removing X₂). In other words, you can isolate the partial effect of any variable by first "partialling out" the other covariates from both the outcome and the variable of interest, then regressing the resulting residuals on each other. This is the core insight behind modern causal inference methods like DoubleML, which replace the linear regressions in the residualization step with flexible machine learning models while preserving the same two-step logic.

In [ ]:
dag = Digraph()
dag.attr(rankdir="LR")
dag.node("Intelligence", "Intelligence\n(Confounder)")
dag.node("Education", "Years of Education\n(Treatment)")
dag.node("Income", "Income\n(Outcome)")

dag.edge("Intelligence", "Education", label="  1.5")
dag.edge("Intelligence", "Income", label="  2.0")
dag.edge("Education", "Income", label="  3.0 (true effect)")

dag

In [ ]:
# 1. Generate Synthetic Data
np.random.seed(42)
n_samples = 500

# The Confounder ("Intelligence")
intelligence = np.random.normal(10, 2, n_samples)

# The Treatment ("Years of Education")
# Treatment depends on the Confounder
years_edu = 1.5 * intelligence + np.random.normal(0, 1, n_samples)

# The Outcome ("Income")
# Y = 3.0 * Treatment + 2.0 * Confounder + Noise
# The TRUE CAUSAL EFFECT we want to find is 3.0
true_effect = 3.0

income = (
    true_effect * years_edu + 2.0 * intelligence + np.random.normal(0, 2, n_samples)
)

# 2. Naive Regression (Biased)
# Regressing Outcome on Treatment (Ignoring Confounder)
X_naive = sm.add_constant(years_edu)
model_naive = sm.OLS(income, X_naive).fit()
naive_coef = model_naive.params[1]

# 3. Full Regression (The "Correct" Standard Way)
# Regressing Outcome on Treatment + Confounder
X_full = sm.add_constant(np.column_stack([years_edu, intelligence]))
model_full = sm.OLS(income, X_full).fit()
full_coef = model_full.params[1]

# 4. FWL Theorem: Manual Residualization

# Step A: Residualize Outcome (Remove Confounder from Outcome)
# Regress Y ~ Confounder
model_y_conf = sm.OLS(income, sm.add_constant(intelligence)).fit()
res_y = model_y_conf.resid

# Step B: Residualize Treatment (Remove Confounder from Treatment)
# Regress T ~ Confounder
model_t_conf = sm.OLS(years_edu, sm.add_constant(intelligence)).fit()
res_t = model_t_conf.resid

# Step C: Regress Residuals on Residuals
# No constant needed because residuals have mean 0
model_fwl = sm.OLS(res_y, res_t).fit()
fwl_coef = model_fwl.params[0]

# 5. Compare Results
print(f"True causal effect:  {true_effect:.4f}")
print(f"Naive (biased):      {naive_coef:.4f}")
print(f"Full regression:     {full_coef:.4f}")
print(f"FWL (residualized):  {fwl_coef:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Naive regression
x_range_naive = np.linspace(years_edu.min(), years_edu.max(), 100)
sns.scatterplot(x=years_edu, y=income, alpha=0.3, ax=axes[0])
axes[0].plot(
    x_range_naive,
    model_naive.params[0] + naive_coef * x_range_naive,
    color="red",
    label=f"Naive (β={naive_coef:.2f})",
)
axes[0].plot(
    x_range_naive,
    model_naive.params[0] + true_effect * x_range_naive,
    color="green",
    linestyle="--",
    label=f"True effect (β={true_effect:.1f})",
)
axes[0].set(
    xlabel="Years of Education",
    ylabel="Income",
    title="Naive Regression (Omitted Variable Bias)",
)
axes[0].legend()

# Panel 2: Full regression (partial regression plot)
sns.scatterplot(x=years_edu, y=income, alpha=0.3, ax=axes[1])
axes[1].plot(
    x_range_naive,
    model_full.params[0]
    + full_coef * x_range_naive
    + model_full.params[2] * intelligence.mean(),
    color="red",
    label=f"Full (β={full_coef:.2f})",
)
axes[1].plot(
    x_range_naive,
    model_full.params[0]
    + true_effect * x_range_naive
    + model_full.params[2] * intelligence.mean(),
    color="green",
    linestyle="--",
    label=f"True effect (β={true_effect:.1f})",
)
axes[1].set(
    xlabel="Years of Education",
    ylabel="Income",
    title="Full Regression (Controlling for Intelligence)",
)
axes[1].legend()

# Panel 3: FWL residualized regression
x_range_fwl = np.linspace(res_t.min(), res_t.max(), 100)
sns.scatterplot(x=res_t, y=res_y, alpha=0.3, ax=axes[2])
axes[2].plot(
    x_range_fwl, fwl_coef * x_range_fwl, color="red", label=f"FWL (β={fwl_coef:.2f})"
)
axes[2].plot(
    x_range_fwl,
    true_effect * x_range_fwl,
    color="green",
    linestyle="--",
    label=f"True effect (β={true_effect:.1f})",
)
axes[2].set(
    xlabel="Education Residuals",
    ylabel="Income Residuals",
    title="FWL: Residual-on-Residual",
)
axes[2].legend()

plt.tight_layout()
plt.show()

### Double/Debiased ML Example

In [ ]:
# DoubleML from scratch: FWL with Random Forest + cross-fitting
# Cross-fitting avoids overfitting bias — we never predict on training data.

# Reshape confounder (intelligence) into a 2D array for sklearn (n_samples x 1)
X_conf = intelligence.reshape(-1, 1)
# Number of cross-validation folds for the cross-fitting procedure
n_folds = 5
# Create a K-Fold splitter that shuffles data before splitting into 5 folds
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Pre-allocate arrays to store residuals for every observation
res_y_dml = np.zeros(n_samples)  # will hold outcome residuals (income - predicted income)
res_t_dml = np.zeros(n_samples)  # will hold treatment residuals (education - predicted education)

# Loop over each fold: train on K-1 folds, predict on the held-out fold
for train_idx, test_idx in kf.split(X_conf):

    # --- Step 1 ("First ML"): Residualize the OUTCOME ---
    # Train a Random Forest to predict income from confounders only
    rf_y = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
    rf_y.fit(X_conf[train_idx], income[train_idx])  # fit on training fold
    # Compute outcome residuals on held-out fold: actual income minus RF prediction
    res_y_dml[test_idx] = income[test_idx] - rf_y.predict(X_conf[test_idx])

    # --- Step 2 ("Second ML"): Residualize the TREATMENT ---
    # Train a Random Forest to predict education from confounders only
    rf_t = RandomForestRegressor(n_estimators=200, max_depth=5, random_state=42)
    rf_t.fit(X_conf[train_idx], years_edu[train_idx])  # fit on training fold
    # Compute treatment residuals on held-out fold: actual education minus RF prediction
    res_t_dml[test_idx] = years_edu[test_idx] - rf_t.predict(X_conf[test_idx])

# --- Step 3: Regress outcome residuals on treatment residuals (same as FWL Step C) ---
# The coefficient from this regression is the causal effect estimate
# No intercept needed because residuals are mean-zero by construction
model_dml = sm.OLS(res_y_dml, res_t_dml).fit()
dml_coef = model_dml.params[0]  # extract the single coefficient (causal effect estimate)

# Compare all three approaches
print(f"True causal effect:      {true_effect:.4f}")
print(f"OLS FWL estimate:        {fwl_coef:.4f}")   # linear residualization from earlier
print(f"DoubleML (RF) estimate:  {dml_coef:.4f}")    # ML-based residualization

In [ ]:
# Side-by-side comparison: DoubleML (RF residuals) vs FWL (OLS residuals)
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# --- Panel 1: DoubleML residualized regression ---
# Create x-axis range spanning the treatment residuals for the fitted line
x_range_dml = np.linspace(res_t_dml.min(), res_t_dml.max(), 100)
# Scatter plot of RF-based treatment residuals vs RF-based outcome residuals
sns.scatterplot(x=res_t_dml, y=res_y_dml, alpha=0.3, ax=axes[0])
# Overlay the DoubleML estimated regression line (slope = causal effect estimate)
axes[0].plot(
    x_range_dml, dml_coef * x_range_dml, color="red", label=f"DoubleML (β={dml_coef:.2f})"
)
# Overlay the true causal effect line for comparison
axes[0].plot(
    x_range_dml,
    true_effect * x_range_dml,
    color="green",
    linestyle="--",
    label=f"True effect (β={true_effect:.1f})",
)
axes[0].set(
    xlabel="Education Residuals (RF)",    # treatment variation not explained by confounders
    ylabel="Income Residuals (RF)",       # outcome variation not explained by confounders
    title="DoubleML: Residual-on-Residual",
)
axes[0].legend()

# --- Panel 2: FWL (OLS) residualized regression ---
# Create x-axis range spanning the OLS treatment residuals
x_range_fwl = np.linspace(res_t.min(), res_t.max(), 100)
# Scatter plot of OLS-based treatment residuals vs OLS-based outcome residuals
sns.scatterplot(x=res_t, y=res_y, alpha=0.3, ax=axes[1])
# Overlay the FWL estimated regression line
axes[1].plot(
    x_range_fwl, fwl_coef * x_range_fwl, color="red", label=f"FWL (β={fwl_coef:.2f})"
)
# Overlay the true causal effect line for comparison
axes[1].plot(
    x_range_fwl,
    true_effect * x_range_fwl,
    color="green",
    linestyle="--",
    label=f"True effect (β={true_effect:.1f})",
)
axes[1].set(
    xlabel="Education Residuals (OLS)",   # treatment variation not explained by confounders
    ylabel="Income Residuals (OLS)",      # outcome variation not explained by confounders
    title="FWL: Residual-on-Residual",
)
axes[1].legend()

plt.tight_layout()
plt.show()

### 401k data example with DoubleML

What is the effect of 401k participation on financial wealth at different quantiles?

[Based on this DoubleML tutorial](https://docs.doubleml.org/stable/examples/py_double_ml_pension_qte.html)

[Kallus et al. 2019](https://arxiv.org/abs/1912.12945)

In [ ]:
! pip install --quiet doubleml

import doubleml as dml          # DoubleML library for causal inference with ML
import multiprocessing           # to detect CPU cores for parallel model fitting
from doubleml.datasets import fetch_401K  # built-in 401(k) dataset

from sklearn.base import clone   # clone() creates unfitted copies of estimators

# LightGBM: gradient boosting for classification (propensity) and regression (outcome)
from lightgbm import LGBMClassifier, LGBMRegressor

In [ ]:
# Load the 401(k) dataset: treatment is e401 eligibility, outcome is net_tfa (net financial assets)
data = fetch_401K(return_type='DataFrame')

# Preview first 5 rows to inspect the data
display(data.head())

# Show summary statistics (mean, std, min, max, quartiles) for all columns
display(data.describe().round(2))

In [ ]:
# List of confounders/controls (age, income, education, family size, marital status, etc.)
features_base = ["age", "inc", "educ", "fsize", "marr", "twoearn", "db", "pira", "hown"]

# Initialize the DoubleML data-backend object, specifying:
#   y_col: outcome variable (net total financial assets)
#   d_cols: treatment variable (401k eligibility)
#   x_cols: confounders to control for
data_dml_base = dml.DoubleMLData(
    data, y_col="net_tfa", d_cols="e401", x_cols=features_base
)

In [ ]:
# Define quantile grid: 0.10, 0.15, 0.20, ..., 0.90 (17 quantiles)
tau_vec = np.arange(0.1, 0.95, 0.05)

# Number of cross-fitting folds (data is split into 3 parts for cross-fitting)
n_folds = 3

# Classification learner: used for propensity score (P(treatment=1|X)) and quantile indicator
# Small model (50 trees, 8 leaves) to avoid overfitting; verbose=-1 suppresses output
class_learner = LGBMClassifier(n_estimators=50, learning_rate=0.05, num_leaves=8, verbose=-1, n_jobs=1)
# Regression learner: used for outcome modeling (not used directly here but available)
# reg_learner = LGBMRegressor(n_estimators=50, learning_rate=0.05, num_leaves=8, verbose=-1, n_jobs=1)

In [ ]:
# Pre-allocate arrays to store estimated potential quantiles for treatment=0 and treatment=1
PQ_0 = np.full((len(tau_vec)), np.nan)  # potential quantiles under no treatment
PQ_1 = np.full((len(tau_vec)), np.nan)  # potential quantiles under treatment

# Pre-allocate arrays for 95% confidence interval bounds (lower, upper)
ci_PQ_0 = np.full((len(tau_vec), 2), np.nan)  # CI for Y(0) quantiles
ci_PQ_1 = np.full((len(tau_vec), 2), np.nan)  # CI for Y(1) quantiles

# Loop over each quantile level (e.g., 10th percentile, 15th percentile, ...)
for idx_tau, tau in enumerate(tau_vec):
    print(f"Quantile: {tau}")

    # Estimate the potential quantile for the UNTREATED group Y(0) at this quantile
    # ml_g: classifier to model P(Y <= quantile | X, D=0) (conditional distribution)
    # ml_m: classifier to model P(D=1 | X) (propensity score for IPW reweighting)
    # clone() ensures each iteration gets a fresh, unfitted copy of the learner
    dml_PQ_0 = dml.DoubleMLPQ(
        data_dml_base,
        ml_g=clone(class_learner),        # nuisance model for outcome distribution
        ml_m=clone(class_learner),        # nuisance model for propensity score
        score="PQ",                       # potential quantile score function
        treatment=0,                      # estimate quantile of Y(0) (untreated potential outcome)
        quantile=tau,                     # which quantile to estimate (e.g., 0.10 = 10th percentile)
        n_folds=n_folds,                  # number of cross-fitting folds
        normalize_ipw=True,               # normalize inverse probability weights to sum to 1
        trimming_rule="truncate",         # truncate extreme propensity scores...
        trimming_threshold=1e-2,          # ...at 0.01 and 0.99 to avoid instability
    )

    # Estimate the potential quantile for the TREATED group Y(1) at this quantile
    dml_PQ_1 = dml.DoubleMLPQ(
        data_dml_base,
        ml_g=clone(class_learner),        # nuisance model for outcome distribution
        ml_m=clone(class_learner),        # nuisance model for propensity score
        score="PQ",                       # potential quantile score function
        treatment=1,                      # estimate quantile of Y(1) (treated potential outcome)
        quantile=tau,                     # which quantile to estimate
        n_folds=n_folds,                  # number of cross-fitting folds
        normalize_ipw=True,               # normalize inverse probability weights
        trimming_rule="truncate",         # truncate extreme propensity scores
        trimming_threshold=1e-2,          # minimum propensity score threshold
    )

    # Fit both models (runs cross-fitting, nuisance estimation, and debiased estimation)
    dml_PQ_0.fit()
    dml_PQ_1.fit()

    # Store the estimated potential quantile values
    PQ_0[idx_tau] = dml_PQ_0.coef[0]     # point estimate for Y(0) quantile
    PQ_1[idx_tau] = dml_PQ_1.coef[0]     # point estimate for Y(1) quantile

    # Store the 95% confidence interval bounds for each estimate
    ci_PQ_0[idx_tau, :] = dml_PQ_0.confint(level=0.95).to_numpy()  # [lower, upper] for Y(0)
    ci_PQ_1[idx_tau, :] = dml_PQ_1.confint(level=0.95).to_numpy()  # [lower, upper] for Y(1)

In [ ]:
# Print the last fitted DoubleML model summary (shows coefficient, std error, t-stat, p-value, CI)
print(dml_PQ_1)

In [ ]:
# Assemble all potential quantile results into a single DataFrame for display
data_pq = {"Quantile": tau_vec,
           "DML Y(0)": PQ_0, "DML Y(1)": PQ_1,                       # point estimates
           "DML Y(0) lower": ci_PQ_0[:, 0], "DML Y(0) upper": ci_PQ_0[:, 1],  # CI for Y(0)
           "DML Y(1) lower": ci_PQ_1[:, 0], "DML Y(1) upper": ci_PQ_1[:, 1]}  # CI for Y(1)
df_pq = pd.DataFrame(data_pq)
# Display rounded to 2 decimal places
display(df_pq.round(2))

In [ ]:
# --- Plot styling setup ---
sns.set_theme()
colors = sns.color_palette()

plt.rcParams['figure.figsize'] = 10., 7.5
sns.set_theme(font_scale=1.5)
# Clean white grid style with no border spines
sns.set_style('whitegrid', {'axes.spines.top': False,
                            'axes.spines.bottom': False,
                            'axes.spines.left': False,
                            'axes.spines.right': False})

# --- Create side-by-side plots for Y(0) and Y(1) potential quantile functions ---
plt.rcParams['figure.figsize'] = 10., 7.5
fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.grid(visible=True); ax2.grid(visible=True)

# Left panel: Potential quantile function for UNTREATED Y(0)
ax1.plot(df_pq['Quantile'], df_pq['DML Y(0)'], color='violet', label='Estimated Quantile Y(0)')
# Shaded region shows the 95% confidence interval around Y(0) estimates
ax1.fill_between(df_pq['Quantile'], df_pq['DML Y(0) lower'], df_pq['DML Y(0) upper'], color='violet', alpha=.3, label='Confidence Interval')
ax1.legend()

# Right panel: Potential quantile function for TREATED Y(1)
ax2.plot(df_pq['Quantile'], df_pq['DML Y(1)'], color='violet', label='Estimated Quantile Y(1)')
# Shaded region shows the 95% confidence interval around Y(1) estimates
ax2.fill_between(df_pq['Quantile'], df_pq['DML Y(1) lower'], df_pq['DML Y(1) upper'], color='violet', alpha=.3, label='Confidence Interval')
ax2.legend()

# Shared title and axis labels across both panels
fig.suptitle('Potential Quantiles', fontsize=16)
fig.supxlabel('Quantile')
_ = fig.supylabel('Potential Quantile and 95%-CI')

In [ ]:
# Detect available CPU cores and use up to 2 for parallel model fitting
n_cores = multiprocessing.cpu_count()
cores_used = np.min([2, n_cores - 1])  # use at most 2 cores (leave 1 free)
print(f"Number of Cores used: {cores_used}")

np.random.seed(42)

# DoubleMLQTE estimates the Quantile Treatment Effect: QTE(tau) = Q_Y(1)(tau) - Q_Y(0)(tau)
# This does both Y(0) and Y(1) estimation jointly across all quantiles in one object
dml_QTE = dml.DoubleMLQTE(
    data_dml_base,
    ml_g=clone(class_learner),       # nuisance model for conditional outcome distribution
    ml_m=clone(class_learner),       # nuisance model for propensity score
    quantiles=tau_vec,               # vector of quantile levels to estimate (0.10 to 0.90)
    score='PQ',                      # potential quantile score function
    n_folds=n_folds,                 # 3-fold cross-fitting
    normalize_ipw=True,              # normalize IPW weights to sum to 1
    trimming_rule="truncate",        # truncate extreme propensity scores
    trimming_threshold=1e-2,         # minimum propensity score of 0.01
)

# Fit the model; n_jobs_models parallelizes across quantile levels
dml_QTE.fit(n_jobs_models=cores_used)

In [ ]:
# Print QTE summary table: coefficient (QTE), std error, t-stat, p-value, 95% CI for each quantile
print(dml_QTE)

In [ ]:
# Perform multiplier bootstrap (500 reps) to construct JOINT confidence bands
# Joint CIs account for testing multiple quantiles simultaneously (wider than pointwise CIs)
dml_QTE.bootstrap(n_rep_boot=500)
# Compute joint 95% confidence interval using the bootstrap critical values
ci_QTE = dml_QTE.confint(level=0.95, joint=True)

# Assemble QTE results into a DataFrame: point estimates + joint CI bounds
data_qte = {"Quantile": tau_vec, "DML QTE": dml_QTE.coef,
            "DML QTE lower": ci_QTE["2.5 %"], "DML QTE upper": ci_QTE["97.5 %"]}
df_qte = pd.DataFrame(data_qte)
print(df_qte)

In [ ]:
# --- Plot the Quantile Treatment Effect curve with joint confidence band ---
plt.rcParams['figure.figsize'] = 10., 7.5
fig, ax = plt.subplots()
ax.grid(visible=True)

# Plot QTE point estimates across quantiles (how much 401k eligibility shifts each quantile)
ax.plot(df_qte['Quantile'], df_qte['DML QTE'], color='violet', label='Estimated QTE')
# Shaded band shows the joint 95% CI (accounts for multiple testing across quantiles)
ax.fill_between(df_qte['Quantile'], df_qte['DML QTE lower'], df_qte['DML QTE upper'], color='violet', alpha=.3, label='Confidence Interval')

plt.legend()
plt.title('Quantile Treatment Effects', fontsize=16)
plt.xlabel('Quantile')
_ = plt.ylabel('QTE and 95%-CI')